# 01 — Feature Pipeline Consolidation (FeatureBuilder)

**Giai đoạn 2**: Hợp nhất pipeline tạo đặc trưng với nguyên tắc ngăn ngừa rò rỉ dữ liệu (Leakage-Safe) và nhất quán giữa huấn luyện và dự báo (Train–Serve Consistency).

### Mục tiêu:
1. Trải nghiệm `FeatureBuilder` đóng vai trò là nguồn chân lý duy nhất (Single Source of Truth) cho feature engineering.
2. Tách bạch hoàn toàn `fit()` (chỉ học thống kê trên Train) và `transform()` (áp dụng trên Test).
3. Sử dụng `shift(1)` trước các phép tính cửa sổ trượt (rolling) để loại bỏ hoàn toàn lookahead leakage.
4. Xác thực tính nhất quán Train–Serve: `build_single_step()` cho dự báo đệ quy trả về kết quả khớp tuyệt đối với batch `transform()`.


In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np

from src.data.loader import DataLoader, time_series_split
from src.featurengineering.builder import FeatureBuilder, FeatureConfig
from src.featurengineering.stationarity_test import StationarityTester


## 1. Nạp Dữ liệu & Phân chia Canonical Split


In [2]:
df = DataLoader().load_data()
train_df, test_df = time_series_split(df)
print(f"Train size: {train_df.shape}, Test size: {test_df.shape}")


Train size: (7426, 31), Test size: (2191, 31)


## 2. Khởi tạo & Fit FeatureBuilder trên Tập Train

Cấu hình các nhóm đặc trưng:
- **Đặc trưng trễ (Lags)**: Các bước trễ [1, 2, 3, 7] ngày.
- **Đặc trưng thống kê cửa sổ trượt (Rolling)**: Cửa sổ 7, 14, 30 ngày (tính mean, std) — tự động áp dụng `shift(1)` để ngăn ngừa rò rỉ ngày hiện tại.
- **Đặc trưng chu kỳ thời gian (Temporal)**: Sin/Cos mã hóa tháng, ngày trong năm, biến chỉ báo mùa mưa (Is_Wet_Season).
- **Bộ lọc chất lượng (Quality Filter)**: Lọc bỏ đặc trưng phương sai thấp hoặc tương quan quá yếu với target.


In [3]:
config = FeatureConfig(
    lag_columns=["Lượng mưa", "Nhiệt độ 2m", "Độ ẩm tương đối 2m", "Tốc độ gió 2m"],
    lag_periods=[1, 2, 3, 7],
    rolling_columns=["Lượng mưa", "Nhiệt độ 2m"],
    rolling_windows=[7, 14, 30],
    rolling_stats=["mean", "std"],
    temporal_features=["Month_sin", "Month_cos", "DayOfYear_sin", "DayOfYear_cos", "Is_Wet_Season"],
    wet_season_months=[5, 6, 7, 8, 9, 10, 11],
    apply_quality_filter=True,
    target_col="Lượng mưa",
    date_col="Ngày"
)

fb = FeatureBuilder(config)
fb.fit(train_df)
print("✅ FeatureBuilder đã được fit thành công trên tập Train duy nhất.")


📐 FeatureBuilder.fit() — learning from train data only
📐 fit_missing_value_stats: computed fill_mean for 30 columns (train only)
⏰ CREATING LAG FEATURES
   ✅ Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (7426, 47)
🪟 CREATING ROLLING FEATURES
   🔒 Leakage-safe mode: shift(1) applied before rolling
   ✅ Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (7426, 59)
🕒 CREATING TEMPORAL FEATURES
   ✅ Created 5 temporal features
   Dataset shape: (7426, 64)
📐 fit_feature_quality: 62 features analyzed (train only)
   Keep: 55, Drop: 7
   ✅ FeatureBuilder fitted
✅ FeatureBuilder đã được fit thành công trên tập Train duy nhất.


## 3. Batch Transformation trên Tập Train và Tập Test

Áp dụng `transform()` cho cả 2 tập. Các giá trị mean imputation và danh sách cột sau lọc chất lượng được lấy hoàn toàn từ trạng thái đã fit của Train.


In [ ]:
X_train = fb.transform(train_df)
X_test = fb.transform(test_df)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Số lượng đặc trưng tạo ra: {X_train.shape[1] - 2}")  # Trừ Ngày và Lượng mưa
display(X_train.head(3))


## 4. Xác thực Tính Nhất quán Train–Serve (build_single_step Verification)

Trong bài toán dự báo đệ quy (recursive multi-step forecasting), mỗi bước tương lai được dự báo từng ngày một.
Hàm `build_single_step(history)` phải tái sử dụng chính xác logic của `transform()`.
Kiểm tra độ chênh lệch giữa batch `transform()` và `build_single_step()` trên cùng một cửa sổ lịch sử.


In [ ]:
# Lấy cửa sổ 50 ngày cuối của tập Train
test_window = train_df.iloc[-50:].copy()

# Batch transform
batch_tail = fb.transform(test_window).tail(1).reset_index(drop=True)

# Single step transform
single_step = fb.build_single_step(test_window).reset_index(drop=True)

# So sánh giá trị trên toàn bộ các cột số
numeric_cols = batch_tail.select_dtypes(include=[np.number]).columns
max_abs_diff = (batch_tail[numeric_cols] - single_step[numeric_cols]).abs().max().max()

print(f"Độ lệch tối đa giữa transform() và build_single_step(): {max_abs_diff:.2e}")
assert max_abs_diff < 1e-6, "build_single_step phải khớp tuyệt đối với transform!"
print("✅ Xác thực thành công: Train-Serve consistency đạt độ chính xác số học tuyệt đối!")


## 5. Kiểm định Tính Dừng Chuẩn hóa (StationarityTester)

Sử dụng `StationarityTester` duy nhất (`src.featurengineering.stationarity_test`) để kiểm định chuỗi lượng mưa.


In [ ]:
res = StationarityTester.test_stationarity(train_df['Lượng mưa'], regression='c', verbose=True)
print(f"ADF Statistic:  {res['ADF_statistic']:.4f}, p-value: {res['ADF_pvalue']:.4e} -> Dừng: {res['ADF_stationary']}")
print(f"KPSS Statistic: {res['KPSS_statistic']:.4f}, p-value: {res['KPSS_pvalue']:.4e} -> Dừng: {res['KPSS_stationary']}")
print(f"Kết luận tổng thể: {res['conclusion']}")


## 6. Tổng kết Gate Check (Stage 2)

| Tiêu chí | Kết quả | Chi tiết |
|---|---|---|
| Fit/Transform Separation | ✅ PASS | Không rò rỉ dữ liệu Test vào imputation/quality filter |
| Lookahead Leakage Guard | ✅ PASS | `shift(1)` trước mọi rolling window |
| Train-Serve Consistency | ✅ PASS | `build_single_step()` khớp 100% với `transform()` |
| Single Source Stationarity | ✅ PASS | `StationarityTester` đặt tại 1 nơi duy nhất |
